<a href="https://colab.research.google.com/github/RKumarAccount/MyPractice/blob/main/AI_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
!pip install langchain langchain-community pandas pydantic langchain-mistralai

In [28]:
#Importing module

##Impoting module

In [29]:
import os
import csv
from typing import List, Literal
from pydantic import BaseModel, Field
from langchain_mistralai import ChatMistralAI
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain.agents.structured_output import ToolStrategy

In [30]:
from google.colab import userdata
os.environ["MISTRAL_API_KEY"] = userdata.get('MISTRAL_API_KEY')

In [31]:
##Schema


In [32]:
class Task(BaseModel):
  title: str = Field(description="The Task Name")
  priority: Literal["low", "medium", "high"]
  reason: str = Field(description="The reason for the task")

class TaskList(BaseModel):
  task: List[Task]


In [33]:
#priortize the task

@tool
def priortize_task(tasks:List[str])-> List[dict]:
  """Assign priority to tasks"""

  priortitized=[]

  for task in tasks:
    task_lower=task.lower()

    if any(w_ord in task_lower for w_ord in ['urgent','asap','immediately']):
      priority='high'
      reason='conatins urgency keywordss'

    elif any(w_ord in task_lower for w_ord in ['today','soon']):
      priority='medioum'
      reason='time bound not urgent'

    else:
      priority='low'
      reason='not urgency detected'

    priortitized.append({'title':task,'priority':priority,'reason':reason})
  return priortitized

##Save the file

In [34]:
def save_tasks_to_csv(tasks: List[dict], filename: str = "/content/tasks.csv") -> str:
  """Save a list of tasks to a CSV file."""

  fieldNames=['title','priority','reason']

  with open(filename,'w',newline='',encoding='utf-8') as csvfile:
    writer=csv.DictWriter(csvfile,fieldnames=fieldNames)

    high_priority=[task for task in tasks if task['priority']=='high']
    medium_priority=[task for task in tasks if task['priority']=='medium']
    low_priority=[task for task in tasks if task['priority']=='low']

    tasks=high_priority+medium_priority+low_priority

    writer.writeheader()
    for task in tasks:
      writer.writerow(task)

  return f'task successfully save to {filename}'

#call llm

In [35]:
llm=ChatMistralAI(
    model="mistral-small",
    temperature=0
)

#Agent creation


In [36]:
agent=create_agent(
    model=llm,
    tools=[priortize_task,save_tasks_to_csv],
    response_format=ToolStrategy(schema=TaskList)
)

##Run Agent

In [37]:
from langchain_core.messages import HumanMessage

user_input='''
Tomoow I need to finish the report asap,
call my namager,
buy groceries,
and prepare slides urgently for prsentation'''

response=agent.invoke(
    {
        "messages": [
            HumanMessage(
                content=user_input+'\n\n After proritizing save the task into a csv file'
            )
        ]
    }
)